# AgentCore Gateway with Serverless OAuth proxy for API Key auth

## Overview

This notebook deploys a **serverless OAuth proxy** using API Gateway + Lambda,
eliminating the need for developers to run local proxy and callback servers.

### Architecture


### What This Deploys

1. **API Gateway** - Fake IdP
2. **IdP Lambda** - Lambda function that provides Oauth /token endpoint for M2M auth flow returning an API Key.
5. **AgentCore Gateway** - A Gateway with IAM auth that can be used to test the IdP

## Step 1: Setup

In [ ]:
# Install dependencies
!pip3 install -r requirements.txt --quiet

Deploy the CDK stack:

```bash
cd cdk
npm install
cdk deploy
```

then copy the output in the cell below replacing the content.

In [ ]:
import boto3

In [ ]:
output = """
ApiKeyIdpStack.ApiEndpoint = https://lwh7l2dl42.execute-api.eu-west-1.amazonaws.com/
ApiKeyIdpStack.Gateway = apikeyidpstack-agentcore-gateway-nhwlc3ftcs
ApiKeyIdpStack.ProxyLambdaName = ApiKeyIdpStack-IdpLambda0CAAE044-2C6PUeXfjvbA
ApiKeyIdpStack.VSCodeMcpConfig = {
  "servers": {
    "agentcore-confluence": {
      "type": "http",
      "url": "https://lwh7l2dl42.execute-api.eu-west-1.amazonaws.com/mcp",
      "headers": {
        "MCP-Protocol-Version": "2025-11-25"
      }
    }
  }
}
"""

In [ ]:
config = {}
for el in output.split("\n"):
    if "=" in el:
        key, value = el.split("=", 1)
        config[key.strip().replace("ApiKeyIdpStack.", "")] = value.strip()

## Step 1: Create the AgentCore Credential Provider

In [ ]:
import boto3

client = boto3.client('bedrock-agentcore-control')

In [ ]:
credential_provider_name = "OauthAPIKey"

In [ ]:
client.delete_oauth2_credential_provider(name=credential_provider_name)

In [ ]:
provider = client.create_oauth2_credential_provider(
    name=credential_provider_name,
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        'customOauth2ProviderConfig': {
            'clientId': 'abc',
            'clientSecret': 'abc',
            'oauthDiscovery': {
                'authorizationServerMetadata':  {
                    'tokenEndpoint': config['ApiEndpoint']+'token',
                    'issuer': config['ApiEndpoint'],
                    'authorizationEndpoint': config['ApiEndpoint']+'authorization',
                }
            }
        }
    }
)

# Step 2 - Test the provider

In [ ]:
import boto3
sess = boto3.Session()

ac = sess.client("bedrock-agentcore-control")
acr = sess.client('bedrock-agentcore')
ic = sess.client('bedrock-agentcore-control', endpoint_url=acr._endpoint.host)

In [ ]:
identities = ac.list_workload_identities()['workloadIdentities']
identities

In [ ]:
resp = acr.get_workload_access_token_for_user_id(workloadName=identities[0]['name'], userId='massi.ang@gmail.com')


In [ ]:
workload_access_token = resp['workloadAccessToken']

In [ ]:
resp = acr.get_resource_oauth2_token(workloadIdentityToken=workload_access_token, 
                                     resourceCredentialProviderName=credential_provider_name,
                                    scopes=['list', 'invoke', 'openid'],
                                    customParameters={'provider': 'atlassian'},
                              oauth2Flow='M2M')
print("Access Token:")
print(resp['accessToken'][:50])

## Step 3: Create the target

In [ ]:
import json

api_endpoint = config["ApiEndpoint"]
gateway_id = config["Gateway"]
mcp_server_url = "https://mcp.com/"

target_response = client.create_gateway_target(
    name="MCPServer",
    description="API Key authenticated target",
    gatewayIdentifier=gateway_id,
    credentialProviderConfigurations=[{
        "credentialProviderType": "OAUTH",
        "credentialProvider": {
            "oauthCredentialProvider": {
                "providerArn": provider['credentialProviderArn'],
                "grantType": "CLIENT_CREDENTIALS",
                "scopes": [],
                "customParameters": {"provider": "atlassian"}
            }
        }
    }],
    targetConfiguration={"mcp": {"mcpServer": {"endpoint": mcp_server_url} }}
)
target_id = target_response["targetId"]
print(f"✓ Confluence target: {target_id}")

----

## Cleanup (Optional)

Run this cell to delete all resources created by this notebook.

In [ ]:
client = boto3.client('bedrock-agentcore-control')

def cleanup():
    """Delete all resources created by this notebook."""
    print("Cleaning up resources...")
    # Delete Gateway target and gateway
    targets = client.list_gateway_targets(gatewayIdentifier=config["Gateway"])['items']
    try:
        for t in targets:
            print(f" Deleting target {t['targetId']} for gateway {config["Gateway"]}", end='')
            client.delete_gateway_target(gatewayIdentifier=config["Gateway"], targetId=t['targetId'])
            print(" done")

    except: pass
    
    # Delete credential provider
    try:
        client.delete_oauth2_credential_provider(name=credential_provider_name)
        print(f"✓ Deleted Credential Provider")
    except: pass

# Uncomment to run cleanup:
cleanup()

To remove the remaining resources run:

```bash
cd cdk
cdk destroy
```

In [ ]:
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
import boto3

def sign_request(request):
    """Sign an HTTP request with AWS SigV4."""
    session = boto3.Session()
    credentials = session.get_credentials()
    region = session.region_name or "us-east-1"

    aws_request = AWSRequest(
        method=request.get_method(),
        url=request.get_full_url(),
        data=request.data,
        headers=request.headers,
    )
    SigV4Auth(credentials, "bedrock-agentcore", region).add_auth(aws_request)

    # Update original request headers
    for key, value in aws_request.headers.items():
        request.add_header(key, value)

In [ ]:
import json
GATEWAY_URL = 'https://agentcore-mcp-gateway-v0nkcactp1.gateway.bedrock-agentcore.eu-west-1.amazonaws.com/mcp'
from urllib.request import Request, urlopen

body = json.dumps({
  "jsonrpc": "2.0",
  "id": 2,
  "method": "tools/list",
  "params": {}
})
req = Request(GATEWAY_URL, method="POST", headers={"Content-Type": "application/json"}, data=body.encode('utf8'))

sign_request(req)

print(
            "{}\n{}\r\n{}\r\n\r\n{}".format(
                "-----------START-----------",
                (req.method or "GET") + " " + req.full_url,
                "\r\n".join("{}: {}".format(k, v) for k, v in req.headers.items()),
                req.data,
            )
        )

with urlopen(req) as resp:
    resp_body = resp.read().decode()
    print(resp_body)
    print(resp.headers)
